<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
%cd /content
!rm -rf flyrank-ml-internship
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

/content
/content/flyrank-ml-internship
Loaded 30,000 rows


In [25]:
import pandas as pd

staleness_check = (
    df.groupby('freshness_tier')['trend_direction']
      .apply(lambda s: pd.Series({
          'n': len(s),
          'pct_declining': (s == 'down').mean() * 100
      }))
      .unstack()
)

tier_order = ['0-30', '31-90', '91-180', '181+']
staleness_check = staleness_check.reindex(tier_order)
staleness_check['n'] = staleness_check['n'].astype(int)
print(staleness_check.round(1))

                    n  pct_declining
freshness_tier                      
0-30            20480           51.1
31-90             175           58.9
91-180           9171           61.1
181+              174           47.1


In [26]:
position_check = (
    df.groupby('position_tier')['ctr']
      .agg(n='count', median_ctr='median', mean_ctr='mean')
)

tier_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep', 'no_data']
position_check = position_check.reindex([t for t in tier_order if t in position_check.index])
print(position_check.round(2))

                   n  median_ctr  mean_ctr
position_tier                             
top_3           2321        0.00      1.48
page_1         11814        0.16      0.65
striking        7304        0.11      0.32
page_3_5        7242        0.03      0.22
deep            1319        0.00      0.15


In [27]:
top3_rows = df[df['position_tier'] == 'top_3']
print(f"top_3 rows: {len(top3_rows)}")
print(f"of those, avg_position == 0: {(top3_rows['avg_position'] == 0).sum()}")
print(top3_rows.groupby(top3_rows['avg_position'] == 0)['ctr'].median())

top_3 rows: 2321
of those, avg_position == 0: 1205
avg_position
False    0.0
True     0.0
Name: ctr, dtype: float64


In [28]:
df['position_tier_clean'] = df['position_tier'].where(
    df['avg_position'] != 0, 'no_position_data'
)

position_check_clean = (
    df.groupby('position_tier_clean')['ctr']
      .agg(n='count', median_ctr='median', mean_ctr='mean')
)

tier_order = ['no_position_data', 'top_3', 'page_1', 'striking', 'page_3_5', 'deep']
position_check_clean = position_check_clean.reindex(
    [t for t in tier_order if t in position_check_clean.index]
)
print(position_check_clean.round(2))

                         n  median_ctr  mean_ctr
position_tier_clean                             
no_position_data      1205        0.00      0.30
top_3                 1116        0.00      2.76
page_1               11814        0.16      0.65
striking              7304        0.11      0.32
page_3_5              7242        0.03      0.22
deep                  1319        0.00      0.15


In [29]:
top3_clean = df[(df['position_tier'] == 'top_3') & (df['avg_position'] != 0)]
print(top3_clean['impressions_90d'].describe())

count      1116.000000
mean       6299.900538
std       29321.214397
min           1.000000
25%           3.000000
50%          53.000000
75%        2698.250000
max      509252.000000
Name: impressions_90d, dtype: float64


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Verdict — staleness → decline rate: MIXED.** Decline rate rises through 0-30 → 31-90 →
91-180 (51.1% → 58.9% → 61.1%), supporting the belief in that range, but reverses at 181+
(47.1%, n=174 — thin enough not to trust fully). Confirmed for the middle of the range only.

**Verdict — CTR vs. position peers: CONFIRMED.** Median and mean CTR fall cleanly and
monotonically from page_1 through deep, with no exceptions. The apparent exception at top_3
was explained by (a) a position_tier construction bug sweeping avg_position==0 ("no data")
rows into the top tier, and (b) even after fixing that, a volume artifact (25th percentile
impressions = 3) — not a refutation of the position-comparison principle.

**In one sentence, the way the session builds a flag:**
Among visible content pages (impressions_90d ≥ 500 — a floor set by the CTR-vs-position
check, which showed that low-volume pages produce meaningless CTR/rate numbers), look at
whether the page is currently declining (trend_direction == "down") and moderately-to-very
stale (freshness_tier in {31-90, 91-180} days since last update). If both are true, flag it
for refresh review.

**Why this population gate:** Signal check 2 showed CTR (and by extension, other rate-based
metrics) is unreliable below a minimum volume — a page with 3 impressions can swing from 0%
to 33% CTR on a single click. 500 impressions/90d is a conservative floor to keep the rule
off thin-evidence pages.

**Why this staleness window, not "any staleness":** Signal check 1 (staleness → decline rate)
came back MIXED — the decline rate does rise through 0-30 → 31-90 → 91-180 (51% → 59% → 61%),
supporting the belief in that range, but reverses at 181+ (down to 47%, on a thin n=174).
Rather than extrapolate a rule off a tail I don't trust, the condition only fires in the
range where the pattern actually held: 31-90 and 91-180 days stale. Pages older than that
are not automatically excluded from other treatment, just from *this* rule.

**Why declining, not just stale:** Staleness alone (from signal check 1) is only weakly
predictive on its own — the effect is real but not large, and a rule that flags every stale
page regardless of outcome would flood the queue (slide 8's lesson: a queue that can't say
"no action" gets ignored). Requiring an observed decline alongside staleness narrows the flag
to pages where there's actual evidence of a problem, not just an old timestamp.

**Reason code (one):** `STALE_DECLINING_VISIBLE` — travels with every flagged row so a
reviewer sees why the system raised its hand, not just that it did.

**Action:** `refresh` — update the content, refresh examples/data, revisit the title's year
if relevant. This matches Article C's treatment from the session (the "fading star" pattern).

**Ranking / score:** Not every flagged page is equally urgent. I rank by what's at stake —
impressions currently being reached, scaled by how much the trend has already dropped — so a
high-visibility page in steep decline outranks a low-visibility page in mild decline, even if
both cleared the same yes/no flag.

**What this rule does NOT claim:** flagging a page as stale-and-declining is not proof a
refresh will fix it — consolidation, seasonality, and SERP changes (per the lane guide's
decline-vs-lookalikes table) can all produce the same pattern. This is a decision-support
flag, not a diagnosis.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [30]:
# Cell: Build the rule and rank the queue

MIN_IMPRESSIONS = 500
STALE_TIERS = ['31-90', '91-180']

# Population gate — from the CTR/volume lesson in signal check 2
eligible = df[df['impressions_90d'] >= MIN_IMPRESSIONS].copy()

# Condition — from signal check 1, only the range where staleness→decline actually held
is_declining = eligible['trend_direction'] == 'down'
is_stale = eligible['freshness_tier'].isin(STALE_TIERS)

eligible['flagged'] = is_declining & is_stale
flagged = eligible[eligible['flagged']].copy()

flagged['reason_code'] = 'STALE_DECLINING_VISIBLE'
flagged['action'] = 'refresh'

# Score — what's at stake: impressions currently reached, scaled by how far
# the trend has already dropped (slide 11's ranking logic)
flagged['score'] = flagged['impressions_90d'] * (flagged['trend_pct'].abs() / 100)

queue = flagged.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = [
    'content_id', 'client_id', 'score', 'reason_code', 'action',
    'impressions_90d', 'trend_pct', 'trend_direction',
    'freshness_tier', 'days_since_last_update', 'ctr', 'avg_position'
]
queue_out = queue[output_cols]

import os
os.makedirs('work/outputs', exist_ok=True)
queue_out.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Eligible (impressions_90d >= {MIN_IMPRESSIONS}): {len(eligible):,}")
print(f"Flagged (STALE_DECLINING_VISIBLE): {len(flagged):,}")
print(f"Share of eligible flagged: {len(flagged) / len(eligible):.1%}")
queue_out.head(20)

Eligible (impressions_90d >= 500): 16,726
Flagged (STALE_DECLINING_VISIBLE): 4,083
Share of eligible flagged: 24.4%


,content_id,client_id,score,reason_code,action,impressions_90d,trend_pct,trend_direction,freshness_tier,days_since_last_update,ctr,avg_position
0,content_5fe46e04994d,client_4e07408562,231936.320,STALE_DECLINING_VISIBLE,refresh,517715,-44.8,down,91-180,104,0.14,4.2
1,content_cb112fce36be,client_19581e27de,129542.380,STALE_DECLINING_VISIBLE,refresh,309910,-41.8,down,91-180,104,0.16,5.6
2,content_2c2606c5d176,client_19581e27de,126800.635,STALE_DECLINING_VISIBLE,refresh,347399,-36.5,down,91-180,104,0.53,4.2
3,content_9532f197bbc8,client_4e07408562,115328.616,STALE_DECLINING_VISIBLE,refresh,309192,-37.3,down,91-180,104,0.87,2.0
4,content_3d94572c3a35,client_19581e27de,99886.452,STALE_DECLINING_VISIBLE,refresh,190623,-52.4,down,91-180,104,0.24,4.3
5,content_124763d39ca5,client_6208ef0f77,94885.993,STALE_DECLINING_VISIBLE,refresh,129803,-73.1,down,91-180,104,0.01,33.2
6,content_c8e9d6ab9013,client_19581e27de,90566.252,STALE_DECLINING_VISIBLE,refresh,208678,-43.4,down,91-180,104,0.00,9.7
7,content_8b36799b7e44,client_6208ef0f77,88657.800,STALE_DECLINING_VISIBLE,refresh,141400,-62.7,down,91-180,104,0.02,32.0
8,content_89fcb6f35525,client_6208ef0f77,79704.456,STALE_DECLINING_VISIBLE,refresh,174408,-45.7,down,91-180,104,0.56,4.7
9,content_813e88069237,client_6208ef0f77,78943.618,STALE_DECLINING_VISIBLE,refresh,233561,-33.8,down,91-180,104,0.06,26.2


**Note on Section 2's score, before Section 3:** the ranking above (`STALE_DECLINING_VISIBLE`,
`score = impressions_90d * abs(trend_pct)/100`) is for prioritizing the flagged queue only —
it is not the Precision@K baseline used to compare against Week 5 (see the decline-blind
scoring below), since it uses `trend_direction`/`trend_pct` directly to decide both membership
and rank, which the Week 5 model will not be allowed to do.


In [31]:
print(flagged['days_since_last_update'].value_counts().head(10))
print(flagged['days_since_last_update'].describe())

days_since_last_update
104    3967
106      58
41       29
89        7
102       5
60        3
105       3
34        3
96        2
40        1
Name: count, dtype: int64
count    4083.000000
mean      103.405094
std         6.061420
min        34.000000
25%       104.000000
50%       104.000000
75%       104.000000
max       106.000000
Name: days_since_last_update, dtype: float64


**Finding:** 97.2% of flagged rows share `days_since_last_update == 104` exactly.
This is very unlikely to be organic — almost certainly a synthetic-data batch
artifact (per DATA_USE.md's "realistic but synthetic" framing) rather than a
real-world staleness distribution. The rule's staleness condition is doing its
job correctly against this data, but this concentration means the staleness
half of the flag carries less real diagnostic weight here than the decline
half does — worth caveating in the top-20 review and treating as a data
limitation, not a rule flaw.

# Precision@K baseline metric (compare against base rate, not in isolation)

In [32]:
import os
print(os.getcwd())
print(os.path.exists('scripts/ml_utils.py'))

/content/flyrank-ml-internship
True


In [34]:
# Cell: Precision@K baseline metric — decline-blind ranking score
# (must not use trend_direction / trend_pct to decide who's eligible for top-K,
#  or the metric becomes tautological — see w04 Precision@K writeup)

import sys, os
sys.path.append(os.path.abspath('scripts'))
from ml_utils import precision_at_k

eligible['is_declining_label'] = (eligible['trend_direction'] == 'down').astype(int)

# Decline-blind score: staleness + visibility only.
# days_since_last_update rewards staler pages; impressions_90d rewards visible ones.
# No trend_direction, no trend_pct anywhere in this formula or in what's eligible.
eligible['staleness_visibility_score'] = (
    eligible['days_since_last_update'] * eligible['impressions_90d']
)

base_rate = eligible['is_declining_label'].mean()
p20 = precision_at_k(eligible['is_declining_label'], eligible['staleness_visibility_score'], k=20)
p50 = precision_at_k(eligible['is_declining_label'], eligible['staleness_visibility_score'], k=50)

print(f"Eligible population base rate (declining share): {base_rate:.3f}")
print(f"Baseline Precision@20 (staleness+visibility, decline-blind): {p20:.3f}")
print(f"Baseline Precision@50 (staleness+visibility, decline-blind): {p50:.3f}")

Eligible population base rate (declining share): 0.596
Baseline Precision@20 (staleness+visibility, decline-blind): 0.450
Baseline Precision@50 (staleness+visibility, decline-blind): 0.440


**v1 result — worse than random.** Precision@20 = 0.450 and Precision@50 = 0.440, both
*below* the 0.596 eligible-population base rate. A ranking that guessed randomly from this
pool would expect ~0.596 in its top-K by chance; this score underperforms that. That's a
real, meaningful negative result about this particular formula — not a bug to explain away —
but it's worth understanding *why* before drawing conclusions about staleness itself, since
the formula multiplies two very differently-scaled numbers together (checked below).


In [35]:
print(eligible['days_since_last_update'].value_counts().head(5))

days_since_last_update
104    6450
20     5225
22     2015
14      486
25      441
Name: count, dtype: int64


In [36]:
print(eligible[['staleness_visibility_score', 'impressions_90d']].corr())

                            staleness_visibility_score  impressions_90d
staleness_visibility_score                    1.000000         0.790845
impressions_90d                               0.790845         1.000000


**Confound confirmed: this is mostly an impressions ranking wearing a staleness costume.**
`staleness_visibility_score = days_since_last_update * impressions_90d` correlates 0.79 with
`impressions_90d` alone. `days_since_last_update` ranges roughly 0–300; `impressions_90d`
ranges into the hundreds of thousands — the product is dominated almost entirely by the
larger-scale term. So v1's 0.44 isn't really testing "does staleness predict decline" — it's
mostly testing "does ranking by raw impressions predict decline" (answer: no, slightly worse
than the base rate here). That's a distinct, real finding, but not the one Section 1's signal
check was actually asking about. Fixing the scale mismatch with rank-based scoring, below.


In [37]:
eligible['staleness_visibility_score_v2'] = (
    eligible['days_since_last_update'].rank(pct=True) +
    eligible['impressions_90d'].rank(pct=True)
)

p20_v2 = precision_at_k(eligible['is_declining_label'], eligible['staleness_visibility_score_v2'], k=20)
p50_v2 = precision_at_k(eligible['is_declining_label'], eligible['staleness_visibility_score_v2'], k=50)

print(f"Baseline Precision@20 (rank-based, scale-corrected): {p20_v2:.3f}")
print(f"Baseline Precision@50 (rank-based, scale-corrected): {p50_v2:.3f}")

Baseline Precision@20 (rank-based, scale-corrected): 0.800
Baseline Precision@50 (rank-based, scale-corrected): 0.560


NOTE — how staleness_visibility_score_v2 works:

rank(pct=True) replaces each row's raw value with its percentile position
(0-1) within that column, discarding the original scale entirely.
  e.g. days_since_last_update: 300 becomes ~1.0, 10 becomes ~0.2, 5 becomes ~0.0
  same operation independently on impressions_90d

This fixes v1's scale confound: raw days_since_last_update (roughly 0-300)
was swamped by raw impressions_90d (roughly 0-500K) when multiplied
directly, which is why v1 correlated 0.79 with impressions_90d alone.
Once both columns are 0-1 percentiles, neither can dominate by sheer
magnitude — summing them means each contributes roughly equally to the
combined score.

Why precision still degrades at K=50 despite the fix:
rank() assigns tied values the average rank of the tied block. 38.6% of
eligible rows (confirmed: eligible['days_since_last_update']==104 gives
0.386) share the exact value 104 — a synthetic-data artifact, not real
staleness variation. Those roughly 6,450 tied rows all land near the same
staleness rank (about 0.50), so once you're past the small set of
genuinely-stale outliers (top 20, which spanned 104-194), staleness stops
differentiating and impressions_90d's rank does all the remaining work —
reintroducing the v1 confound one layer deeper. That's the mechanism
behind 0.800 dropping to 0.560.

Takeaway: rank-based scoring fixes a scale problem, not a lack-of-variation
problem. A column that's 38.6% identical values can't be rescued by
percentile ranking alone.

In [38]:
ranked_v2 = eligible.sort_values('staleness_visibility_score_v2', ascending=False).reset_index(drop=True)
print(ranked_v2.loc[:19, ['days_since_last_update', 'impressions_90d', 'is_declining_label']].describe())
print(ranked_v2.loc[20:49, ['days_since_last_update', 'impressions_90d', 'is_declining_label']].describe())

       days_since_last_update  impressions_90d  is_declining_label
count               20.000000        20.000000           20.000000
mean               125.200000    133222.150000            0.800000
std                 36.618373    166808.644171            0.410391
min                104.000000     12029.000000            0.000000
25%                104.000000     21707.750000            1.000000
50%                106.000000     34152.500000            1.000000
75%                117.250000    298620.750000            1.000000
max                194.000000    517715.000000            1.000000
       days_since_last_update  impressions_90d  is_declining_label
count               30.000000        30.000000           30.000000
mean               104.066667    171455.933333            0.400000
std                  0.365148     44083.249090            0.498273
min                104.000000     10910.000000            0.000000
25%                104.000000    149697.500000            0.00

In [39]:
print((eligible['days_since_last_update'] == 104).mean())
print(eligible['days_since_last_update'].rank(pct=True).describe())

0.3856271672844673
count    16726.000000
mean         0.500030
std          0.275437
min          0.000060
25%          0.285543
50%          0.501973
75%          0.801716
max          0.999970
Name: days_since_last_update, dtype: float64


**v2 result — real signal at the top, collapsing to artifact by K=50.** Rank-based scoring
(percentile rank of staleness + percentile rank of visibility, summed) removes the scale
confound and gives Precision@20 = 0.800 (well above the 0.596 base rate — genuine signal) and
Precision@50 = 0.560 (below base rate — signal doesn't hold at depth).

The top-20 vs. rows-21-50 breakdown explains the split:
- **Top 20:** `days_since_last_update` genuinely spans 104–194 (mean 125.2, 75th pct 117.25) —
  real staleness variation, not stuck at the synthetic floor. Combined with high impressions,
  this is the strongest joint signal in the eligible population, and precision reflects that.
- **Rows 21–50:** `days_since_last_update` collapses almost entirely to the `104`-tied mass
  (mean 104.07, std 0.37) — confirmed separately that 38.6% of the full eligible population
  shares that exact value (the same synthetic-batch artifact flagged in Section 2). With
  staleness no longer differentiating anything inside that block, `rank(pct=True)` assigns
  those tied rows nearly the same score (~0.50 median), and `impressions_90d` re-dominates the
  ranking — reintroducing v1's confound, just one layer deeper.

**Takeaway, and a general lesson for Week 5:** staleness carries real predictive signal, but
only where it isn't swamped by the synthetic tie at 104 — this baseline is meaningfully useful
as a top-20 tool and not as a top-50 one. More broadly: percentile-rank features degrade hard
when a column has heavy value concentration, since `.rank(pct=True)` gives tied values nearly
identical scores regardless of how many rows share them. Worth remembering when building
features for the Week 5 model, not just here.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Action for every row: `refresh`. Reason code for every row: `STALE_DECLINING_VISIBLE`.**
Since action and reason code are constant across the queue by construction, this review
focuses on differentiated confidence and what would make each recommendation wrong.

**Two clusters hiding inside one reason code.** Looking at `avg_position` and `ctr` together,
the top 20 splits into two distinct situations that a single reason code doesn't distinguish:

- **Cluster A — ranking intact, demand dropped** (rows 0, 1, 2, 3, 4, 6, 8, 14, 15):
  `avg_position` is 2–10 (page one), CTR is non-trivial (0.14–0.87). Google still ranks
  these pages well; something else pulled impressions down.
- **Cluster B — ranking itself has slipped** (rows 5, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19):
  `avg_position` is 22–47 (page 2–4+), CTR is near-zero (0.00–0.10) — expected given that
  position, not surprising on its own.

This matters for the treatment: Cluster A's problem is probably not staleness at all — the
page still ranks fine, so a refresh may not touch whatever actually caused the drop. Cluster
B's problem is more plausibly content-related, but "refresh" alone may not undo a real
ranking loss if something structural (competition, backlinks) is behind it.

| # | Confidence note | What would make this wrong |
|---|---|---|
| 0 | High — huge scale (518K imp), steep drop, position still strong (4.2) | Ranking's fine; likely a demand or consolidation issue, not a content problem — refresh may not move it |
| 1 | High — large scale, position strong (5.6), decent CTR | Same as row 0 — good ranking argues against a content-quality cause |
| 2 | High — large scale, best CTR in top 20 (0.53) at strong position | Page is clearly working when found; drop is more likely seasonal or SERP-feature (AI overview) loss |
| 3 | High — best CTR (0.87) and best position (2.0) in the set | This page is performing well by every visible metric except trend — strongest case for a lookalike (seasonality/consolidation), not real decline |
| 4 | High — steep drop (-52%), strong position (4.3) | Same pattern as row 0 — ranking intact, cause likely external to content |
| 5 | Medium — steepest drop so far (-73%) but position weak (33.2), CTR ~0 | Ranking itself may already be the primary problem; refresh treats staleness, may not fix a position-driven decline |
| 6 | Medium — decent scale, position okay (9.7) but CTR is exactly 0 | Zero CTR at position 9.7 is unusually low for that position — worth checking if this page even has a valid snippet, separate from staleness |
| 7 | Medium — steep drop (-63%) but position weak (32.0), CTR ~0 | Same as row 5 — page 3 position may be the real story, not freshness |
| 8 | High — good scale, strong position (4.7), solid CTR | Ranking intact — lookalike risk (seasonality/consolidation) same as rows 0-4 |
| 9 | Medium — moderate drop (-34%, weakest decline in top 20), position weak (26.2) | Smallest decline signal here — closest to noise; worth a manual look before committing effort |
| 10 | Medium — moderate scale/drop, position weak (22.1) | Similar to row 9 — page-3 position may predate this window, not caused by recent staleness |
| 11 | Medium — steep drop (-55%), position very weak (47.0), CTR ~0 | Deep page-4+ position — refresh unlikely to be sufficient alone; may need a bigger structural fix |
| 12 | Medium — steepest overall drop (-83%) but smallest impressions base (84K) in top 20, position weak (45.6) | Same story as row 11, plus: with the smallest base here, this rank is sensitive to how much weight trend_pct carries vs impressions_90d in the score |
| 13 | Medium — steep drop (-53%), position weak (23.9), CTR ~0.01 | Ranking already poor — question is whether staleness caused the position loss or just correlates with it |
| 14 | High — strong position (6.2), decent scale | Ranking intact — same lookalike risk as rows 0-4, 8 |
| 15 | High — strong position (6.4), decent scale | Same as row 14 |
| 16 | Medium — steep drop (-66%), position weak (28.6) | Page-3 position; refresh addresses content freshness but not necessarily the ranking gap itself |
| 17 | Medium — steep drop (-52%), position weak (27.2) | Same as row 16 |
| 18 | Medium — steep drop (-54%), position weak (34.4) | Same pattern, deeper page-4 territory |
| 19 | Medium — steep drop (-65%), position weak (38.5), smallest scale near bottom of top 20 | Same pattern; also the row where score is most sensitive to trend_pct weighting given lower impressions base |

**Client concentration.** Only 3 distinct clients appear across the entire top 20:
`client_6208ef0f77` (12/20 rows — and all of Cluster B, rows 5, 7, 9-13, 16-19),
`client_19581e27de` (6/20), `client_4e07408562` (2/20). This is not necessarily a flaw —
these may genuinely be the largest, highest-traffic content libraries in the dataset, which
an impressions-weighted score will naturally surface first. But it's worth naming as a limit:
the score may be crowding out real problems at smaller clients whose declines are proportionally
larger but numerically smaller, and it means `client_6208ef0f77`'s entire Cluster B cluster was
surfaced together by shared scale, not independently vetted row by row. A per-client cap or
separate client-level pass would be a reasonable next iteration.

**Staleness caveat, applying to all 20 rows.** 97.2% of the full flagged queue (3,967 of 4,083
rows) shares `days_since_last_update == 104` exactly — see the finding logged after the queue-build
cell. This is almost certainly a synthetic-data batch artifact rather than a real-world staleness
distribution, so the staleness half of `STALE_DECLINING_VISIBLE` carries less independent
diagnostic weight here than the decline half does, for every row in this review.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks in the top 20:**
- **Row 3** — best CTR (0.87) and best position (2.0) in the set, yet flagged as declining.
  A page performing this well when found is the weakest case for "content is the problem";
  more likely seasonality or consolidation absorbing its demand, per the lane guide's
  decline-vs-lookalike table.
- **Row 9** — smallest decline in the top 20 (-33.8%, just past the -20% threshold that
  defines `trend_direction == "down"`), riding into the queue mainly on impression scale
  rather than strength of decline evidence.
- **Row 12** — smallest impressions base (84K) in the top 20; its rank depends more on the
  steep -83% trend than on scale, making it the row most sensitive to how the score formula
  weights decline magnitude vs. raw impressions.

**Leakage / product-flag check:**
- The rule uses exactly four columns: `impressions_90d`, `trend_pct`, `trend_direction`,
  `freshness_tier` — all observed-window signals, none from a future window.
- `trend_pct`/`trend_direction` are the label source for `is_declining_label`, but this is a
  hand-written rule, not a trained model predicting that label — using them directly here
  matches the starter pipeline's own `declining_with_demand` baseline reason code, and is not
  the same leakage case as w03's `imp_last15` trap (which fed a *future* window into a model
  trained to predict a *same-window* label).
- No FlyRank product flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`)
  are present in this dataset at all, per `DATA_USE.md` — so there is nothing from that
  category to accidentally leak in.
- `content_id` / `client_id` appear only in the output for identification/grouping; confirmed
  they are not part of the score formula (`impressions_90d * abs(trend_pct)/100`).

In [40]:
# Cell: leakage / product-flag audit
score_inputs = ['impressions_90d', 'trend_pct', 'trend_direction', 'freshness_tier']
print("Columns used in the rule:", score_inputs)
print()
print("None of these are FlyRank product flags — health_score, priority_score, action_type,")
print("and needs_ctr_fix are not present in this dataset at all (per DATA_USE.md), so there is")
print("nothing to accidentally leak in from them.")
print()
print("content_id / client_id used only for identity and grouping in the output —")
print("confirming they were never part of the score calculation:")
print("score formula: impressions_90d * abs(trend_pct)/100")

Columns used in the rule: ['impressions_90d', 'trend_pct', 'trend_direction', 'freshness_tier']

None of these are FlyRank product flags — health_score, priority_score, action_type,
and needs_ctr_fix are not present in this dataset at all (per DATA_USE.md), so there is
nothing to accidentally leak in from them.

content_id / client_id used only for identity and grouping in the output —
confirming they were never part of the score calculation:
score formula: impressions_90d * abs(trend_pct)/100


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.